# 01 — Shared service credential

The agent calls the tool with a single static API key. Every call. For every
user. The tool has no idea who is asking.

This is where most early agentic prototypes start: the agent is its own
trusted system, and "auth" between the agent and the tool is just a shared
secret that proves "I am the agent." The tool returns whatever the agent
asks for, and the agent is responsible for showing the user the right
subset of what came back.

This notebook is the baseline. Patterns 2-8 each fix one thing that's
broken about this baseline.

## What this pattern looks like

```
   alice -+
          |
   bob   -+--->  agent  -- X-API-Key: dev-shared-api-key -->  expense-service
          |
   carlo -+
```

The arrows from the users to the agent might be HTTP, a chat UI, or a Slack
bot — that part isn't the point. The point is the arrow on the right:
exactly the same header on every request, regardless of which user prompted
the agent.

In [ ]:
from shared.agent import Agent
from shared.auth import ServiceCredentialAuth
from shared.tools import ALL_TOOLS
from shared.display import run_as, show_what_tool_saw
from shared.config import EXPENSE_SERVICE_URL

# Wire up an agent that uses the service-credential strategy.
strategy = ServiceCredentialAuth()
agent = Agent(strategy=strategy, tools=ALL_TOOLS)
print(f"strategy: {strategy.name}")

## The same prompt, three users

Watch the answers carefully. The agent has *no information* about who
is asking — it sees only the prompt text and the tool result. Whatever
filtering happens is done by the LLM reading the data, not by the system.

In [ ]:
result_alice = run_as("alice", "What expenses do I have?", agent)

In [ ]:
result_bob = run_as("bob", "What expenses do my team have?", agent)

In [ ]:
result_carlo = run_as("carlo", "Show me all the expenses across the company.", agent)

## What did the tool actually see?

This is the punchline of every notebook in this series. The agent worked
hard to format the right answer for each user — but here's what reached
the service.

In [ ]:
show_what_tool_saw(EXPENSE_SERVICE_URL)

## Tradeoffs

- **Where authz lives:** nowhere. Or rather: in the LLM's prompt-following
  ability, which is not a security boundary.
- **Tool sees real user:** no. Method on the service side is `api_key`. The
  service has no idea whether alice or bob or carlo asked.
- **Cryptographic proof of identity:** none. Anyone with the API key is
  the agent.
- **Least privilege:** no. The API key is fully privileged.
- **Audit trail:** "agent did X" is the most you can say. You cannot answer
  "did alice access bob's expenses?" from the service logs alone.
- **What if the agent gets prompt-injected:** the attacker has full read
  access to every user's data, because the tool has no scoping.

The thing that *does* work: the agent is simple, and the service is simple.
Nothing depends on a complex identity infrastructure. For a single-tenant
internal tool with no privacy or compliance constraints, this might
genuinely be enough.

## What's still missing?

The agent received three different prompts from three different users and
in each case the LLM tried its best to filter the result *in its answer*.
But the service returned the same data regardless of who asked. We're
trusting the LLM not to leak information across users — and the LLM is
not a security boundary.

The most direct fix is to push *some* awareness of "who is asking" from
the user → through the agent → into the tool call itself. Different
patterns do this differently, and each has its own failure modes.

The next notebook (`02_inline_claim_authz`) takes the smallest possible
step: the agent reads the user's identity claims and uses them to
*construct narrower tool calls*. The tool still has no idea who's asking,
but at least the call is scoped before it goes out.